In [ ]:
!pip install playwright pandas
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 MB 60.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 17.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [playwright]
199.3 MiB [] 0% 0.0s199.3 MiB [] 0% 281.7s199.3 MiB [] 0% 306.7s199.3 MiB [] 0% 910.9s199.3 MiB [] 0% 766.0s199.3 MiB [] 0% 1038.9s199.3 MiB [] 0% 920.8s199.3 MiB [] 0% 834.7s199.3 MiB [] 0% 773.3s199.3 MiB [] 0% 722.7s199.3 MiB [] 0% 715.4s199.3 MiB [] 0% 672.3s199.3 MiB [] 0% 642.7s199.3 MiB [] 0% 618.7s199.3 MiB [] 0% 596.3s199.3 MiB [] 0% 574.4s199.3 MiB [] 0% 555.1s199.3 MiB [] 0% 535.9s199.3 MiB [] 0% 518.1s199.3 MiB [] 0% 502.2s199.3 MiB [] 0% 468.9s199.3 MiB [] 0% 440.2s199.3 MiB [] 0% 415.6s199.3 MiB [] 0% 394.7s199.3 MiB [] 0% 376.2s199.3 MiB [] 0% 360.0s199.3 MiB [] 0% 345.5s199.3 MiB [] 0% 332.2s199.3 MiB [] 0% 329.7s199.3 MiB [] 0% 310.1s199.3 MiB [] 0% 294.3s199.3 MiB [] 0% 279.7s199.3 MiB [] 0% 266.6s199.3 MiB [] 0% 253.8s199.3 MiB [] 0% 240.6s199.3 MiB [] 0% 

In [ ]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

async def scrape_ulasan_shopee(product_url, max_halaman=10):
    data_ulasan = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800}
        )
        page = await context.new_page()

        # Fungsi penyadap data JSON ulasan di balik layar
        async def handle_response(response):
            if "get_ratings" in response.url and response.status == 200:
                try:
                    json_data = await response.json()
                    ratings = json_data.get("data", {}).get("ratings", [])
                    for r in ratings:
                        komentar = r.get("comment", "").strip()
                        # Hanya ambil ulasan yang ada teks komentarnya (penting untuk sentimen)
                        if komentar:
                            data_ulasan.append({
                                "Username": r.get("author_username", ""),
                                "Rating": r.get("rating_star", 0),
                                "Komentar": komentar.replace("\n", " "),
                                "Variasi": r.get("product_items", [{}])[0].get("model_name", "") if r.get("product_items") else ""
                            })
                except Exception:
                    pass

        page.on("response", handle_response)

        print(f"Membuka produk: {product_url}")
        await page.goto(product_url, timeout=60000)
        await asyncio.sleep(4)

        # Scroll ke bawah agar bagian "Penilaian Produk" termuat
        for _ in range(5):
            await page.mouse.wheel(0, 1000)
            await asyncio.sleep(1.5)

        # Klik tombol halaman berikutnya (Next) pada bagian ulasan
        for halaman in range(1, max_halaman + 1):
            print(f"Mengambil ulasan halaman {halaman}... (Total terkumpul: {len(data_ulasan)})")
            next_btn = await page.query_selector("button.shopee-icon-button--right")

            if next_btn:
                await next_btn.click()
                await asyncio.sleep(3) # Jeda agar tidak diblokir
            else:
                print("Tombol halaman selanjutnya tidak ditemukan (atau halaman terblokir login).")
                await page.screenshot(path="debug_ulasan.png")
                break

        await browser.close()

    if data_ulasan:
        df = pd.DataFrame(data_ulasan).drop_duplicates(subset=["Username", "Komentar"])
        filename = "dataset_sentimen_rok_shopee.csv"
        df.to_csv(filename, index=False)
        print(f"\nSelesai! {len(df)} ulasan berhasil disimpan ke '{filename}'")
        return df
    else:
        print("\nTidak ada ulasan yang berhasil diambil. Cek 'debug_ulasan.png' untuk memastikan tidak terblokir login.")
        return None

# Ganti URL di bawah ini dengan link produk rok pilihan Anda
url_produk = "https://shopee.co.id/LINK-PRODUK-ROK-ANDA-DISINI"
df_hasil = await scrape_ulasan_shopee(url_produk, max_halaman=10)

ModuleNotFoundError: No module named 'playwright'

In [ ]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

COOKIE_STRING = r"""_gcl_au=1.1.1479015966.1795088599; _gcl_gs=2.1.k1$i1795088580$u81964244; _med=cpc; _sapid=cd758ca21a9572c8c82840a099254e8e5debcf0963408c51f6de2a1a; csrftoken=tz4zFG3t4GNbeMcE2pYnRhpEpN2t9sRp; ssr-tz=Asia/Shanghai; _fbp=fb.2.1795088611835.48117280447967772; SPC_SI=YavEagAAAABaZnNaM3BmNUtFOgIAAAAANTA4NzZxcm4=; SPC_SEC_SI=v1-V2NnUkFpejVqVGxkUjJ6RXemt4pldQTx9rCU6NLDKUWS+CihRlKZWbT5idBSb81t7a49pEVLsWxaN/DP0KQFAJ6lLSblDt+JetVXckRYXWU=; SPC_F=dcYdkt0WIGWLFVTm2VHMH3hmBrnMqE30; REC_T_ID=730845aa-e40e-11f1-aed9-cacd07be71fe; _QPWSDCXHZQA=0cd27905-b6cf-4495-8632-5891159e1699; REC7iLP4Q=249462d7-05c0-4f49-b100-9b0dba95e84e; _gcl_aw=GCL.1795088627.Cj0KCQiA2frXBhC_ARIsAPTFcdOhobRBjNeJXe_wIlyJQ3_5UT3CPhu7pfz20aPsaMVVDQqrUe-xXrcaAgFUEALw_wcB; _ga=GA1.1.1287419665.1795088627; language=id; SPC_CLIENTID=ZGNZZGt0MFdJR1dMjykvjdbbqgyriirm; SPC_ST=PTkGL4olRogEBtmrDQDO67qFttGEetIZ1hlOpKUPP7CBZ4ji4EbIgip/u//lrYmSSEFKCm4qeNrnUhDhWPrPG54nbY+io2iI0021CfAunpbR9+6Gv4g9v7g8JY1hhUfJjO6ZqFhhsKUd7b2tNwxsj8cppB3B5ttNkZwiACJEDVgj+pC/kcNesuWMq6CYKBYj60ySxkxupvx5lefZ+Y0FSA==.AHSGb0MMP6FnK1I4kXJkQEUYQ+muFOJO4udtkBLvaxMw; SPC_U=1020070476; SPC_R_T_ID=tzxRFRFWU86sMWq9EKWqpZS1GfX9XJhx3O3ydAoYQx3OSKgvATxxWpy+XycsCYZ2mvd/0uHyLIKccqAzXbcIk27Hhbj0V/4qagFVJyUqIbBTSCIhiI8hXr1Dqdm+Ek2q8Ar7z2tyC+yecz4voD2H7WbLMWzvDYeIwsr1u25htoE=; SPC_R_T_IV=Qnlvbk5vYVVmY2tMc3IyZg==; SPC_T_ID=tzxRFRFWU86sMWq9EKWqpZS1GfX9XJhx3O3ydAoYQx3OSKgvATxxWpy+XycsCYZ2mvd/0uHyLIKccqAzXbcIk27Hhbj0V/4qagFVJyUqIbBTSCIhiI8hXr1Dqdm+Ek2q8Ar7z2tyC+yecz4voD2H7WbLMWzvDYeIwsr1u25htoE=; SPC_T_IV=Qnlvbk5vYVVmY2tMc3IyZg==; sense_sa_r=s; _ga_SW6D8G0HXK=GS2.1.s1795088626$o1$g1$t1795089571$j32$l1$h1099257803; SPC_CDS_CHAT=91bd8364-5dc6-45f8-9c2a-59f6d4abf6d1; shopee_webUnique_ccd=YhMwCgPldxSESpa9pSoNiQ%3D%3D%7CE%2FENGhfp56oaHkjEhqmhV%2FVbE3umINd0k87z%2FE%2F1%2B0wgQYmeETV%2Btyx0qGlDc3nuZdX5id%2B4YVsTwDY%3D%7CL3ztFyVRg7bmNnTM%7C08%7C3; ds=d1d810cac98b81cee8c101cac0a69601"""

PRODUCT_URL = "https://shopee.co.id/Audy-Skirt-Korean-Loose-Maxi-Skirt-Rok-Wanita-Anti-Kusut-Polyester-Rok-Flare-Wanita-Rok-Basic-Polos-Maxi-Rok-Panjang-Wanita-i.733348959.26969986169"

def parse_cookie_string(cookie_str):
    cookies = []
    for item in cookie_str.split(";"):
        if "=" in item:
            name, value = item.strip().split("=", 1)
            cookies.append({
                "name": name.strip(),
                "value": value.strip(),
                "domain": ".shopee.co.id",
                "path": "/"
            })
    return cookies

async def scrape_audy_skirt(max_halaman=20):
    data_ulasan = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800}
        )

        cookies = parse_cookie_string(COOKIE_STRING)
        await context.add_cookies(cookies)
        print(f"Berhasil memuat {len(cookies)} cookies.")

        page = await context.new_page()

        # Menyadap data JSON ulasan di balik layar
        async def handle_response(response):
            if "get_ratings" in response.url and response.status == 200:
                try:
                    json_data = await response.json()
                    ratings = json_data.get("data", {}).get("ratings", [])
                    for r in ratings:
                        komentar = r.get("comment", "").strip()
                        if komentar:
                            data_ulasan.append({
                                "Username": r.get("author_username", ""),
                                "Rating": r.get("rating_star", 0),
                                "Komentar": komentar.replace("\n", " "),
                                "Variasi": r.get("product_items", [{}])[0].get("model_name", "") if r.get("product_items") else ""
                            })
                except Exception:
                    pass

        page.on("response", handle_response)

        print("Membuka halaman produk Audy Skirt...")
        await page.goto(PRODUCT_URL, timeout=60000)
        await asyncio.sleep(5)

        # Scroll ke bawah agar bagian Penilaian Produk termuat
        for _ in range(5):
            await page.mouse.wheel(0, 1000)
            await asyncio.sleep(1.5)

        # Pindah halaman ulasan satu per satu
        for halaman in range(1, max_halaman + 1):
            print(f"Halaman {halaman}/{max_halaman} | Terkumpul: {len(data_ulasan)} ulasan")
            next_btn = await page.query_selector("button.shopee-icon-button--right")
            if next_btn:
                await next_btn.click()
                await asyncio.sleep(3)
            else:
                print("Tombol Next tidak ditemukan (ulasan habis atau sesi butuh verifikasi).")
                await page.screenshot(path="debug_hasil.png")
                break

        await browser.close()

    if data_ulasan:
        df = pd.DataFrame(data_ulasan).drop_duplicates(subset=["Username", "Komentar"])
        filename = "ulasan_audy_skirt_shopee.csv"
        df.to_csv(filename, index=False)
        print(f"\nBerhasil! {len(df)} ulasan disimpan ke '{filename}'.")
        return df
    else:
        print("\nBelum ada ulasan yang terkumpul. Cek file 'debug_hasil.png' di panel kiri.")
        return None

# Ubah angka max_halaman jika ingin mengambil lebih banyak (1 halaman = sekitar 6 ulasan)
df_hasil = await scrape_audy_skirt(max_halaman=20)

ModuleNotFoundError: No module named 'playwright'

In [ ]:
import requests
import time
import pandas as pd

COOKIE_STRING = r"""_gcl_au=1.1.1479015966.1795088599; _gcl_gs=2.1.k1$i1795088580$u81964244; _med=cpc; _sapid=cd758ca21a9572c8c82840a099254e8e5debcf0963408c51f6de2a1a; csrftoken=tz4zFG3t4GNbeMcE2pYnRhpEpN2t9sRp; ssr-tz=Asia/Shanghai; _fbp=fb.2.1795088611835.48117280447967772; SPC_SI=YavEagAAAABaZnNaM3BmNUtFOgIAAAAANTA4NzZxcm4=; SPC_SEC_SI=v1-V2NnUkFpejVqVGxkUjJ6RXemt4pldQTx9rCU6NLDKUWS+CihRlKZWbT5idBSb81t7a49pEVLsWxaN/DP0KQFAJ6lLSblDt+JetVXckRYXWU=; SPC_F=dcYdkt0WIGWLFVTm2VHMH3hmBrnMqE30; REC_T_ID=730845aa-e40e-11f1-aed9-cacd07be71fe; _QPWSDCXHZQA=0cd27905-b6cf-4495-8632-5891159e1699; REC7iLP4Q=249462d7-05c0-4f49-b100-9b0dba95e84e; _gcl_aw=GCL.1795088627.Cj0KCQiA2frXBhC_ARIsAPTFcdOhobRBjNeJXe_wIlyJQ3_5UT3CPhu7pfz20aPsaMVVDQqrUe-xXrcaAgFUEALw_wcB; _ga=GA1.1.1287419665.1795088627; language=id; SPC_CLIENTID=ZGNZZGt0MFdJR1dMjykvjdbbqgyriirm; SPC_ST=PTkGL4olRogEBtmrDQDO67qFttGEetIZ1hlOpKUPP7CBZ4ji4EbIgip/u//lrYmSSEFKCm4qeNrnUhDhWPrPG54nbY+io2iI0021CfAunpbR9+6Gv4g9v7g8JY1hhUfJjO6ZqFhhsKUd7b2tNwxsj8cppB3B5ttNkZwiACJEDVgj+pC/kcNesuWMq6CYKBYj60ySxkxupvx5lefZ+Y0FSA==.AHSGb0MMP6FnK1I4kXJkQEUYQ+muFOJO4udtkBLvaxMw; SPC_U=1020070476; SPC_R_T_ID=tzxRFRFWU86sMWq9EKWqpZS1GfX9XJhx3O3ydAoYQx3OSKgvATxxWpy+XycsCYZ2mvd/0uHyLIKccqAzXbcIk27Hhbj0V/4qagFVJyUqIbBTSCIhiI8hXr1Dqdm+Ek2q8Ar7z2tyC+yecz4voD2H7WbLMWzvDYeIwsr1u25htoE=; SPC_R_T_IV=Qnlvbk5vYVVmY2tMc3IyZg==; SPC_T_ID=tzxRFRFWU86sMWq9EKWqpZS1GfX9XJhx3O3ydAoYQx3OSKgvATxxWpy+XycsCYZ2mvd/0uHyLIKccqAzXbcIk27Hhbj0V/4qagFVJyUqIbBTSCIhiI8hXr1Dqdm+Ek2q8Ar7z2tyC+yecz4voD2H7WbLMWzvDYeIwsr1u25htoE=; SPC_T_IV=Qnlvbk5vYVVmY2tMc3IyZg==; sense_sa_r=s; _ga_SW6D8G0HXK=GS2.1.s1795088626$o1$g1$t1795089571$j32$l1$h1099257803; SPC_CDS_CHAT=91bd8364-5dc6-45f8-9c2a-59f6d4abf6d1; shopee_webUnique_ccd=YhMwCgPldxSESpa9pSoNiQ%3D%3D%7CE%2FENGhfp56oaHkjEhqmhV%2FVbE3umINd0k87z%2FE%2F1%2B0wgQYmeETV%2Btyx0qGlDc3nuZdX5id%2B4YVsTwDY%3D%7CL3ztFyVRg7bmNnTM%7C08%7C3; ds=d1d810cac98b81cee8c101cac0a69601"""

shop_id = 733348959
item_id = 26969986169
jumlah_ambil = 200  # Ubah sesuai kebutuhan (kelipatan 20)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": f"https://shopee.co.id/product/{shop_id}/{item_id}",
    "Cookie": COOKIE_STRING,
    "X-Requested-With": "XMLHttpRequest",
    "Accept": "application/json"
}

data_ulasan = []

for offset in range(0, jumlah_ambil, 20):
    # filter=1 artinya hanya mengambil ulasan yang ada komentarnya
    url = f"https://shopee.co.id/api/v2/item/get_ratings?filter=1&flag=1&itemid={item_id}&limit=20&offset={offset}&shopid={shop_id}&type=0"

    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        print(f"Diblokir server (Status: {resp.status_code})")
        break

    json_data = resp.json()
    ratings = json_data.get("data", {}).get("ratings")

    if not ratings:
        print("Respons kosong / terkena proteksi anti-bot Shopee:", json_data.get("error_msg", json_data))
        break

    for r in ratings:
        komentar = (r.get("comment") or "").strip()
        if komentar:
            data_ulasan.append({
                "Username": r.get("author_username", ""),
                "Rating": r.get("rating_star", 0),
                "Komentar": komentar.replace("\n", " "),
                "Variasi": r.get("product_items", [{}])[0].get("model_name", "") if r.get("product_items") else ""
            })

    print(f"Offset {offset} selesai | Total terkumpul: {len(data_ulasan)} ulasan")
    time.sleep(2)

if data_ulasan:
    df = pd.DataFrame(data_ulasan)
    df.to_csv("ulasan_audy_skirt_shopee.csv", index=False)
    print(f"\nBerhasil! {len(df)} ulasan disimpan ke 'ulasan_audy_skirt_shopee.csv'")
    display(df.head())

Diblokir server (Status: 403)
